# TP2 - Primer Parcial
Probabilidad y Estadística para la IA (CEIA - UBA)

Matías Souto

## Punto 1 - Estimación por Máxima Verosimilitud (MLE)

Variable aleatoria discreta $X \in \{0,1,2,3\}$ con distribución de probabilidad parametrizada por $\theta$:

| X | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| p(X) | $\theta/3$ | $2\theta/3$ | $(1-\theta)/3$ | $2(1-\theta)/3$ |

Muestra observada: $(0, 0, 0, 3, 0, 0, 2, 2, 0, 3)$

Vamos a construir la función de verosimilitud $L(\theta) = \prod_i p(x_i;\theta)$, obtener la log-verosimilitud, derivar respecto a $\theta$ con `sympy`, y resolver el punto crítico para hallar $\hat\theta_{MLE}$.

In [1]:
import sympy as sp
from collections import Counter

# Muestra observada
muestra = [0, 0, 0, 3, 0, 0, 2, 2, 0, 3]

# Frecuencias absolutas por valor de X
frecuencias = Counter(muestra)
n0 = frecuencias[0]
n1 = frecuencias[1]
n2 = frecuencias[2]
n3 = frecuencias[3]

n0, n1, n2, n3

(6, 0, 2, 2)

In [2]:
theta = sp.symbols('theta', positive=True)

# Probabilidades del modelo
p0 = theta / 3
p1 = 2 * theta / 3
p2 = (1 - theta) / 3
p3 = 2 * (1 - theta) / 3

# Función de verosimilitud L(theta) = producto de p(x_i; theta)
L = p0**n0 * p1**n1 * p2**n2 * p3**n3

# Log-verosimilitud armada como suma de n_i * log(p_i) (evita ambigüedades de signo al simplificar potencias)
log_L = n0 * sp.log(p0) + n1 * sp.log(p1) + n2 * sp.log(p2) + n3 * sp.log(p3)
log_L = sp.expand_log(log_L, force=True)

L, log_L

(theta**6*(1/3 - theta/3)**2*(2/3 - 2*theta/3)**2/729,
 6*log(theta) + 2*log(1/3 - theta/3) + 2*log(2/3 - 2*theta/3) - 6*log(3))

In [3]:
# Derivada de la log-verosimilitud respecto a theta
d_log_L = sp.diff(log_L, theta)
d_log_L_simplificada = sp.simplify(d_log_L)

# Punto crítico: igualar la derivada a 0 y resolver theta en (0,1)
soluciones = sp.solve(sp.Eq(d_log_L_simplificada, 0), theta)
theta_mle = [s for s in soluciones if s.is_real and 0 < s < 1][0]

d_log_L_simplificada, soluciones, theta_mle

(2*(5*theta - 3)/(theta*(theta - 1)), [3/5], 3/5)

In [4]:
# Verificamos que theta_mle es un máximo: la segunda derivada debe ser negativa en ese punto
d2_log_L = sp.diff(log_L, theta, 2)
segunda_derivada_en_mle = d2_log_L.subs(theta, theta_mle)

print(f"Segunda derivada en theta_mle: {segunda_derivada_en_mle} (negativa => máximo)")
print(f"theta_MLE = {theta_mle} = {float(theta_mle)}")

Segunda derivada en theta_mle: -125/3 (negativa => máximo)
theta_MLE = 3/5 = 0.6


### Punto 2: Estimadores de Mínimos Cuadrados (modelo cuadrático)

Datos de producción Y (miles de toneladas) en función del tiempo X (meses):

| X | 1 | 8 | 10 | 16 | 23 |
|---|---|---|----|----|----|
| Y | 15 | 41 | 122 | 294 | 433 |

Modelo: $Y = a + bX + cX^2$

La suma de errores al cuadrado:

$$S(a,b,c) = \sum_i \left(y_i - \hat{y}_i\right)^2 = \sum_i \left(y_i - a - bx_i - cx_i^2\right)^2$$

la vamos a derivar parcialmente respecto de cada parámetro e igualar a 0.

**Derivada respecto de $a$:**

$$\frac{\partial S}{\partial a} = \sum_i 2\left(y_i - a - bx_i - cx_i^2\right)(-1) = 0 \;\; \Rightarrow \;\; \sum_i y_i = na + b\sum_i x_i + c\sum_i x_i^2$$

**Derivada respecto de $b$:**

$$\frac{\partial S}{\partial b} = \sum_i 2\left(y_i - a - bx_i - cx_i^2\right)(-x_i) = 0 \;\; \Rightarrow \;\; \sum_i x_iy_i = a\sum_i x_i + b\sum_i x_i^2 + c\sum_i x_i^3$$

**Derivada respecto de $c$:**

$$\frac{\partial S}{\partial c} = \sum_i 2\left(y_i - a - bx_i - cx_i^2\right)(-x_i^2) = 0 \;\; \Rightarrow \;\; \sum_i x_i^2y_i = a\sum_i x_i^2 + b\sum_i x_i^3 + c\sum_i x_i^4$$

Cada ecuación es, literalmente, "$\partial S/\partial a = 0$", "$\partial S/\partial b = 0$" y "$\partial S/\partial c = 0$" reordenadas. Juntas forman el sistema de 3 ecuaciones con 3 incógnitas (a, b, c):

$$\begin{cases} \dfrac{\partial S}{\partial a}=0: & \sum y_i = n\,a + b\sum x_i + c\sum x_i^2 \\[4pt] \dfrac{\partial S}{\partial b}=0: & \sum x_i y_i = a\sum x_i + b\sum x_i^2 + c\sum x_i^3 \\[4pt] \dfrac{\partial S}{\partial c}=0: & \sum x_i^2 y_i = a\sum x_i^2 + b\sum x_i^3 + c\sum x_i^4 \end{cases}$$

Calculamos las sumatorias con los datos y resolvemos el sistema con `sympy.solve`.

In [5]:
import numpy as np

x = np.array([1, 8, 10, 16, 23])
y = np.array([15, 41, 122, 294, 433])

# Matriz de diseño: columnas [1, x, x^2]
X = np.column_stack([np.ones_like(x), x, x**2])

X, y

(array([[  1,   1,   1],
        [  1,   8,  64],
        [  1,  10, 100],
        [  1,  16, 256],
        [  1,  23, 529]]),
 array([ 15,  41, 122, 294, 433]))

In [6]:
# Sumatorias necesarias (a partir de los mismos datos x, y del Punto 2)
n_datos = len(x)
suma_x = np.sum(x)
suma_x2 = np.sum(x**2)
suma_x3 = np.sum(x**3)
suma_x4 = np.sum(x**4)
suma_y = np.sum(y)
suma_xy = np.sum(x * y)
suma_x2y = np.sum(x**2 * y)

# Incógnitas del sistema
a_s, b_s, c_s = sp.symbols('a b c')

# Las 3 ecuaciones normales, escritas explícitamente como sistema
eq1 = sp.Eq(n_datos * a_s + suma_x * b_s + suma_x2 * c_s, suma_y)
eq2 = sp.Eq(suma_x * a_s + suma_x2 * b_s + suma_x3 * c_s, suma_xy)
eq3 = sp.Eq(suma_x2 * a_s + suma_x3 * b_s + suma_x4 * c_s, suma_x2y)

solucion_sistema = sp.solve([eq1, eq2, eq3], [a_s, b_s, c_s])
eq1, eq2, eq3, solucion_sistema

(Eq(5*a + 58*b + 950*c, 905),
 Eq(58*a + 950*b + 17776*c, 16226),
 Eq(950*a + 17776*b + 359474*c, 319160),
 {a: -15604541/1653357, b: 4251710/551119, c: 878435/1653357})

## Punto 2 (Extra): Resolución matricial
Datos de producción Y (miles de toneladas) en función del tiempo X (meses):

| X | 1 | 8 | 10 | 16 | 23 |
|---|---|---|----|----|----|
| Y | 15 | 41 | 122 | 294 | 433 |

Modelo: $Y = a + bX + cX^2$

Para cada observación $(x_i, y_i)$ el modelo predice $\hat{y}_i = a + bx_i + cx_i^2$. En notación matricial, con:

$$X = \begin{pmatrix} 1 & x_1 & x_1^2 \\ 1 & x_2 & x_2^2 \\ \vdots & \vdots & \vdots \\ 1 & x_n & x_n^2 \end{pmatrix}, \quad \beta = \begin{pmatrix} a \\ b \\ c \end{pmatrix}, \quad Y = \begin{pmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{pmatrix}$$

el modelo es $Y = X\beta + \varepsilon$. El estimador de mínimos cuadrados minimiza $\sum \varepsilon_i^2 = (Y - X\beta)^T(Y - X\beta)$, y su solución son las **ecuaciones normales**:

$$X^T X \, \beta = X^T Y \quad \Rightarrow \quad \hat\beta = (X^T X)^{-1} X^T Y$$



In [7]:
# Ecuaciones normales: X^T X beta = X^T Y
XtX = X.T @ X
XtY = X.T @ y

print("X^T X =\n", XtX)
print("\nX^T Y =\n", XtY)

# Resolvemos el sistema lineal para obtener beta = [a, b, c]
beta = np.linalg.solve(XtX, XtY)
a, b, c = beta

print(f"\na = {a:.4f}, b = {b:.4f}, c = {c:.4f}")

X^T X =
 [[     5     58    950]
 [    58    950  17776]
 [   950  17776 359474]]

X^T Y =
 [   905  16226 319160]

a = -9.4381, b = 7.7147, c = 0.5313


In [8]:
# Verificación: comparamos valores predichos por el modelo ajustado contra los observados
y_pred = X @ beta

for xi, yi, ypi in zip(x, y, y_pred):
    print(f"x={xi:>3}  y_real={yi:>4}  y_pred={ypi:8.2f}")

x=  1  y_real=  15  y_pred=   -1.19
x=  8  y_real=  41  y_pred=   86.28
x= 10  y_real= 122  y_pred=  120.84
x= 16  y_real= 294  y_pred=  250.01
x= 23  y_real= 433  y_pred=  449.06


## Punto 3 - Inferencia Bayesiana (distribución posterior de p)

Datos: de 10 clientes a crédito, 2 están en mora. Modelamos la cantidad de clientes en mora como una Binomial(n=10, p), donde p es la proporción de morosidad. El prior sobre p es Beta(α=1, β=2).

Por el teorema de Bayes:

$$\text{posterior}(p) \propto \text{verosimilitud}(p) \times \text{prior}(p) = \text{Binomial}(k=2; n=10, p) \times \text{Beta}(p; \alpha=1, \beta=2)$$

### Derivación simbólica con `sympy`

Multiplicamos verosimilitud × probabilidad a priori, normalizamos integrando, y llegamos a la probabilidad a posteriori.

In [9]:
p = sp.symbols('p', positive=True)

n_clientes = 10
k_mora = 2
alpha_prior = 1
beta_prior = 2

# Verosimilitud: Binomial(k=2; n=10, p). La constante binomial(n,k) no depende de p,
# así que la omitimos (no afecta dónde queda la posterior, sólo su normalización).
verosimilitud = p**k_mora * (1 - p)**(n_clientes - k_mora)

# Prior Beta(alpha, beta): Beta(p; a, b) proporcional a p^(a-1) * (1-p)^(b-1)
prior = p**(alpha_prior - 1) * (1 - p)**(beta_prior - 1)

# Numerador de la posterior (sin normalizar), dejando la base (1-p) intacta
posterior_sin_normalizar = p**(k_mora + alpha_prior - 1) * (1 - p)**(n_clientes - k_mora + beta_prior - 1)
posterior_sin_normalizar

p**2*(1 - p)**9

In [10]:
# Constante de normalización: integramos el numerador entre 0 y 1
constante_normalizacion = sp.integrate(posterior_sin_normalizar, (p, 0, 1))

# Posterior normalizada
posterior_p = posterior_sin_normalizar / constante_normalizacion

# Parámetros de la Beta posterior (por conjugación deberían ser alpha_post = k+alpha_prior, beta_post = n-k+beta_prior)
alpha_post = k_mora + alpha_prior
beta_post = n_clientes - k_mora + beta_prior

constante_normalizacion, posterior_p, (alpha_post, beta_post)

(1/660, 660*p**2*(1 - p)**9, (3, 10))

In [11]:
# Media y varianza de la posterior integrando directamente sobre su densidad: E[p] y E[p^2]
media_posterior_sym = sp.integrate(p * posterior_p, (p, 0, 1))
segundo_momento_sym = sp.integrate(p**2 * posterior_p, (p, 0, 1))
varianza_posterior_sym = sp.simplify(segundo_momento_sym - media_posterior_sym**2)

media_posterior_sym, varianza_posterior_sym

(3/13, 15/1183)